Lokal ineference with ollama and OpenAI gpt-oss.

In [1]:
%pip install jsonschema
%pip install langchain
%pip install langchain-ollama
%pip install pandas
%pip install aijsondbpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


We will create an **agent** that can **answer questions based on data** found in a **JSON file**.
**LangChain** will be used as the **agent framework**.

For the agent, we need:
- A **system prompt**
- The **tool(s)** the agent will use

Let’s start with the **tool**.

To query the JSON data with **pure JavaScript**, we use the **aijsondb** database, which allows querying JSON data using **pure JavaScript**. The **aijsondb** database is initialized with the **sample JSON data** and the **JSON schema** located in the **data folder**.

In [2]:
import aijsondb
aijsondb.init_db("./data/500 KB_V3.json","./data/employeeSchemaDescription_V3.json")

Now we can query the database. It contains **201 employee records**. The number of employees can be **obtained** by:
```
var result=data.employees.length
```


In [3]:
aijsondb.query_data_javascript("var result=data.employees.length")

201


The JavaScript query engine is working. The next step is to create a function that can be used for tool calls.

In [4]:
from langchain.tools import tool
last_answer_from_tool=None
@tool
def query_json_javascript(query: str) -> any:
    """Run a javascript expression on the provided data."""
    global last_answer_from_tool
    try:
        last_answer_from_tool=matches=None
        matches=aijsondb.query_data_javascript(query)
        last_answer_from_tool=matches
        return matches
    except Exception as e:  
        serr=f"Error running JSONPath query: {e}"
        return serr 

Next, we create the **system prompt** to guide the agent. This prompt will include:
- Instructions on how to use the tool
- A **schema description** of the data, so the agent can understand the structure of the data it is working with.

In [5]:
import json
with open('../data/talktodata/employeeSchemaDescription_V3.json') as f:
    jschema = json.load(f)  

schema = json.dumps(jschema, indent=4)

template = f"""
I have a JSON data object.

The JSON schema for the data object you will work with is:
{schema}

Fetch data from the JSON object using javascript to query the data.

Use the tool: 
query_json_javascript: returns the entities using a given javascript programm as query.

Only use the tool if you can be sure from the schema that the query can give correct results.

The data are in a javascript variable named: data.
The final result is always saved in variable result: eg. var result.

Always use the available tools to fetch data.
Only use the data from tools defined above to answer the question.
"""

Now that we have the **tool function** and the **system prompt** for the agent, we can create the agent.

We will use the **OpenAI model**: `gpt-oss-20b` with ollama for the agent. 

Please the  `base_url` to the IP and port of your ollama server.

In [20]:
from langchain.agents import create_agent
from langchain_ollama import ChatOllama

llm =  ChatOllama(
    model="qwen3.6:35b-a3b",
    #"gpt-oss:20b",
    temperature=0,
    max_retries=2,
    base_url="http://192.168.0.124:11434",
    think=False,
    num_ctx=8192
)

agent = create_agent(
    model=llm,
    tools=[query_json_javascript],
    system_prompt=template,
)

That’s it! Now we can ask the agent questions about the data, and it will generate the appropriate **JavaScript queries** to fetch the answers. If a question cannot be answered with the available data, the agent will respond accordingly.

In [21]:
from langchain.messages import HumanMessage 
result = agent.invoke(
    {"messages": [HumanMessage("Hown many employees are there?")]}
)
result['messages'][-1].content

'There are 201 employees in the data.'

The result is correct—there are **201 employees** in the dataset.
The query generated by the LLM is:

In [22]:
result['messages'][1].tool_calls[0]['args']

{'query': 'var result = data.employees.length;'}

The query created by the LLM is identical to the one we used earlier.
Now, let’s see how it performs with a more complex query.

In [23]:
from langchain.messages import HumanMessage 
result = agent.invoke(
    {"messages": [HumanMessage("Which employees have the greatest experience?")]}
)
print(result['messages'][-1].content)

The employees with the greatest experience (10 years) are:

- Eileen Fitzgerald
- Ashley Arroyo
- Blake Mitchell
- Stephanie Lewis
- Chris Thomas
- Brenda Wang
- David Smith
- Patrick Lee
- Jim Werner
- Abigail Briggs
- Cathy Oconnor
- Ricky Perry
- Billy Bradley DVM
- Matthew Caldwell
- Robert Kelley
- Alexander Guerrero IV
- Rebecca King
- Barbara Andrews
- Dustin Newton
- Sara Rivera
- Duane Aguilar
- April Cline
- Robert Johnson
- Willie Jones
- Kelly Cardenas
- Jennifer Anderson
- Mary Fox
- Albert Gonzalez
- Timothy Mullins


In [19]:
result

{'messages': [HumanMessage(content='Which employees have the greatest experience?', additional_kwargs={}, response_metadata={}, id='bb4f4b7c-d6e9-4492-911d-57c3959d69c0'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3.6:35b-a3b', 'created_at': '2026-06-03T10:49:04.574468032Z', 'done': True, 'done_reason': 'length', 'total_duration': 104265444634, 'load_duration': 206847957, 'prompt_eval_count': 512, 'prompt_eval_duration': 4545080000, 'eval_count': 2050, 'eval_duration': 99493118000, 'logprobs': None, 'model_name': 'qwen3.6:35b-a3b', 'model_provider': 'ollama'}, id='lc_run--019e8d18-4be1-7392-b451-ff8f0da24294-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 512, 'output_tokens': 2050, 'total_tokens': 2562})]}

The result is correct: **all employees have 10 years of experience**.
The query is now **more complex**.

In [24]:

args=result['messages'][1].tool_calls[0]['args']
print(args['query'])

var maxExp = 0;
var tempResult = [];

// First pass to find the maximum experience years
data.employees.forEach(emp => {
    if (emp.profile && emp.profile.projects) {
        emp.profile.projects.forEach(proj => {
            if (proj.tasks) {
                proj.tasks.forEach(task => {
                    if (task.assignedTo && task.assignedTo.skills && task.assignedTo.skills.experience) {
                        var years = task.assignedTo.skills.experience.years;
                        if (years > maxExp) {
                            maxExp = years;
                        }
                    }
                });
            }
        });
    }
});

// Second pass to collect employees with the maximum experience
if (maxExp > 0) {
    data.employees.forEach(emp => {
        if (emp.profile && emp.profile.projects) {
            emp.profile.projects.forEach(proj => {
                if (proj.tasks) {
                    proj.tasks.forEach(task => {
                        if (t